## Setup PP-OCRv5

In [1]:
!nvidia-smi

Wed Dec 31 11:45:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install dependencies
!pip install -q paddlepaddle-gpu
!pip install -q paddleocr
!pip install -q scikit-image albumentations lmdb rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 28.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.0/87.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.

In [3]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r /kaggle/working/PaddleOCR/requirements.txt

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 308397, done.
remote: Counting objects: 100% (865/865), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 308397 (delta 753), reused 751 (delta 697), pack-reused 307532 (from 2)
Receiving objects: 100% (308397/308397), 1.63 GiB | 37.38 MiB/s, done.
Resolving deltas: 100% (243974/243974), done.


In [4]:
# Check if paddle installed properly
import paddle
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


I1231 11:48:09.459112    24 program_interpreter.cc:212] New Executor is Running.
W1231 11:48:09.459584    24 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 11.8
W1231 11:48:09.471499    24 gpu_resources.cc:164] device: 0, cuDNN Version: 9.2.


PaddlePaddle works well on 1 GPU.


I1231 11:48:10.284065    24 interpreter_util.cc:624] Standalone Executor is Used.
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_selected_gpus', current_value='1', default_value='')
I1231 11:48:11.810187   106 tcp_utils.cc:107] Retry to connect to 127.0.0.1:52657 while the server is not yet listening.
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_selected_gpus', current_value='0', default_value='')
I1231 11:48:11.812228   105 tcp_utils.cc:181] The server starts to listen on IP_ANY:52657
I1231 11:48:11.812451   105 tcp_utils.cc:130] Successfully connected to 127.0.0.1:52657
I1231 11:48:14.810422   106 tcp_utils.cc:130] Successfully connected to 127.0.0.1:52657
I1231 11:48:14.839838   106 process_group_nccl.cc:129] ProcessGroupNCCL pg_timeout_ 1800000
W1231 11:48:14.842164   106 gpu_resources.cc:119] Please NOTE: device: 1, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 11.

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


I1231 11:48:16.058637   123 tcp_store.cc:289] receive shutdown event and so quit from MasterDaemon run loop


In [5]:
# Download Pretrained Model
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams

--2025-12-31 11:48:16--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105414496 (101M) [application/octet-stream]
Saving to: ‘PP-OCRv5_server_det_pretrained.pdparams’

PP-OCRv5_server_det 100%[===================>] 100.53M  12.4MB/s    in 9.8s    

2025-12-31 11:48:27 (10.3 MB/s) - ‘PP-OCRv5_server_det_pretrained.pdparams’ saved [105414496/105414496]



## Fine-tuning

In [6]:
import os
DATASET_DIR = '/kaggle/input/dataset-nlp-final/Dataset'
TRAIN_DIR = os.path.join(DATASET_DIR, 'Train')
VAL_DIR = os.path.join(DATASET_DIR, 'Val')

TRAIN_GT = os.path.join(TRAIN_DIR, 'det_gt.txt')
VAL_GT = os.path.join(VAL_DIR, 'det_gt.txt')

In [7]:
!python3 -m paddle.distributed.launch --gpus '0,1' PaddleOCR/tools/train.py \
    -c PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
    Train.dataset.data_dir="{TRAIN_DIR}" \
    Train.dataset.label_file_list="['{TRAIN_GT}']" \
    Eval.dataset.data_dir="{VAL_DIR}" \
    Eval.dataset.label_file_list="['{VAL_GT}']" \
    Global.epoch_num=100 \
    Global.eval_batch_step="[0, 500]"

LAUNCH INFO 2025-12-31 11:48:29,193 -----------  Configuration  ----------------------
LAUNCH INFO 2025-12-31 11:48:29,193 auto_parallel_config: None
LAUNCH INFO 2025-12-31 11:48:29,193 auto_tuner_json: None
LAUNCH INFO 2025-12-31 11:48:29,193 devices: 0,1
LAUNCH INFO 2025-12-31 11:48:29,193 elastic_level: -1
LAUNCH INFO 2025-12-31 11:48:29,193 elastic_timeout: 30
LAUNCH INFO 2025-12-31 11:48:29,193 enable_gpu_log: True
LAUNCH INFO 2025-12-31 11:48:29,193 gloo_port: 6767
LAUNCH INFO 2025-12-31 11:48:29,193 host: None
LAUNCH INFO 2025-12-31 11:48:29,193 ips: None
LAUNCH INFO 2025-12-31 11:48:29,193 job_id: default
LAUNCH INFO 2025-12-31 11:48:29,193 legacy: False
LAUNCH INFO 2025-12-31 11:48:29,193 log_dir: log
LAUNCH INFO 2025-12-31 11:48:29,193 log_level: INFO
LAUNCH INFO 2025-12-31 11:48:29,193 log_overwrite: False
LAUNCH INFO 2025-12-31 11:48:29,193 master: None
LAUNCH INFO 2025-12-31 11:48:29,193 max_restart: 3
LAUNCH INFO 2025-12-31 11:48:29,193 nnodes: 1
LAUNCH INFO 2025-12-31 11

In [8]:
!python3 PaddleOCR/tools/export_model.py -c PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml -o \
    Global.pretrained_model=output/PP-OCRv5_server_det/best_accuracy.pdparams \
    Global.save_inference_dir="./PP-OCRv5_server_det_infer/"

Skipping import of the encryption module.
W1231 20:56:55.340369 22992 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 11.8
W1231 20:56:55.342492 22992 gpu_resources.cc:164] device: 0, cuDNN Version: 9.2.
[2025/12/31 20:56:55] ppocr INFO: load pretrain successful from output/PP-OCRv5_server_det/best_accuracy
[2025/12/31 20:56:55] ppocr INFO: Export inference config file to ./PP-OCRv5_server_det_infer/inference.yml
Skipping import of the encryption module
Traceback (most recent call last):
  File "/kaggle/working/PaddleOCR/tools/export_model.py", line 37, in <module>
    main()
  File "/kaggle/working/PaddleOCR/tools/export_model.py", line 33, in main
    export(config)
  File "/kaggle/working/PaddleOCR/ppocr/utils/export_model.py", line 543, in export
    export_single_model(
  File "/kaggle/working/PaddleOCR/ppocr/utils/export_model.py", line 390, in export_single_model
    assert (
           ^
AssertionError
